### 1.1.4. Set Operations

AMPL provides five binary operators that construct new sets from existing ones:
`union`, `inter`, `diff`, `symdiff`, `within` (See section 1_1_3) and `cross`.
These operators may appear anywhere a set expression is valid: in the right-hand side of a
set declaration (`=` or `default`), in an indexing expression, in a `check` statement, or
in a `display` command.

The operators correspond directly to classical set algebra:

| AMPL keyword | Mathematical symbol | Name | Operator precedence (descending — higher number = tighter binding)|
|---|---|---| --- |
| `union` | $A \cup B$ | **Union**: members in either operand | 8* |
| `inter` | $A \cap B$ | **Intersection**: members in both operands | 9* |
| `diff` | $A \setminus B$ | **Difference**: members in left but not right | 8* |
| `symdiff` | $A \triangle B$ | **Symmetric difference**: in one but not both | 8* |
| `cross` | $A \times B$ | **Cartesian product**: All tuples (a, b); generalises to higher dims | 10* |

**All set-valued operators group left-to-right at the same precedence level. Parentheses may always be used to override precedence.*.

**Operand requirements:**

- `union`, `inter`, `diff`, `symdiff`: both operands must be set expressions of the **same dimension**.
  For simple (one-dimensional) sets of strings or numbers, both sides are 1-D.
  For compound sets (pairs, triples), both sides must have matching dimension.
  You cannot, for example, form the union of a set of pairs with a set of triples. 
- `cross`: the operands may have any dimensions; the result has dimension equal to the **sum** of the operand dimensions.
  `A cross B` where `A` is 1-D and `B` is 1-D yields a 2-D set of pairs.
  `A cross B cross C` yields a 3-D set of triples {A, B, C}. It is equivalent to `setof {a in A, b in B, c in C} (a,b,c)`
- `within` / `not within`: both operands are set expressions of the same dimension.
  The result is a **Boolean value** (1 for true, 0 for false), not a new set.

#### Conceptual Explanation

In classical set algebra, given sets $A$ and $B$ drawn from the same universe $U$:

$A \cup B = \{x \in U : x \in A \text{ or } x \in B\}$

$A \cap B = \{x \in U : x \in A \text{ and } x \in B\}$

$A \setminus B = \{x \in U : x \in A \text{ and } x \notin B\}$

$A \triangle B = (A \setminus B) \cup (B \setminus A) = \{x \in U : x \in A \text{ XOR } x \in B\}$

$A \times B = \{(a, b) : a \in A,\ b \in B\}$

AMPL implements these definitions faithfully. The result of every set-valued operator is a
**finite set of distinct elements** — duplicates are never retained. 

**Ordering in results.** The result of `diff` automatically preserves the ordering
of its left operand when that operand is an ordered set — no additional declaration
is required. For example, if `WEEKS` is an ordered set, then 
```python
WEEKS diff {first(WEEKS)}
```
is also ordered, and `prev()` may be called on its members.
By contrast, `union`, `inter`, and `symdiff` do not guarantee preservation of **ordering**; their results should be treated as unordered even if the operands are ordered.

**Role in optimization models.** Set operations serve two primary purposes:

1. **Deriving index sets from data.** Rather than supplying redundant data, define derived
   sets by expressions: `set NUTR = MINREQ union MAXREQ;` automatically includes every
   nutrient that appears in either requirement set, without requiring the modeler to maintain
   a third set manually.

2. **Structuring multi-dimensional index sets.** `cross` and comma-separated indexing
   expressions provide two syntactic paths to the same Cartesian-product index set.
   Named compound sets (defined via `=`) promote clarity in large models by giving the
   product set a stable, reusable name.

#### Operator Reference

The table below documents every set operator.

| Operator | Syntax | Result type | Semantics |
|---|---|---|---|
| `union` | `A union B` | set | Members in `A` or `B` (or both) |
| `inter` | `A inter B` | set | Members in both `A` and `B` |
| `diff` | `A diff B` | set | Members in `A` but not in `B` |
| `symdiff` | `A symdiff B` | set | Members in exactly one of `A`, `B` |
| `cross` | `A cross B` | set (higher dimension) | All ordered pairs (or tuples) from `A` × `B` |
| `union {…}` | `union {i in S} expr[i]` | set | Union of all sets `expr[i]` for `i` in `S`; `{}` when `S` is empty |
| `inter {…}` | `inter {i in S} expr[i]` | set | Intersection of all sets `expr[i]` for `i` in `S`; undefined (runtime error) when `S` is empty |
| `within` | `A within B` | Boolean value (1 for true, 0 for false)| True iff every member of `A` is in `B` |
| `not within` | `A not within B` | Boolean value (1 for true, 0 for false) | Negation of `within` |

Because `union`, `diff`, and `symdiff` have the same precedence, the expression
`A union B diff C` parses as `(A union B) diff C`. The expression
`A union B inter C diff D` parses as `(A union (B inter C)) diff D`
because `inter` binds more tightly than `union` or `diff`.

#### Example 1 — Iterated `union` and `inter`

The iterated forms reduce a collection of sets over an index set,
analogously to `sum` for numeric expressions. They are the standard pattern when the
number of sets to combine is determined by data.

**Empty-index behaviour:** In current AMPL implementations, iterated `inter` over an `{}` (the empty set) produces an error message at model generation time: "*inter over empty set is not defined*". Ensure the index set is non-empty before using iterated `inter`.

In [ ]:
set PROD;
set ORIG {PROD};           # ORIG[p] = set of valid origins for product p
set DEST {PROD};           # DEST[p] = set of valid destinations for product p

set ALL_ORIG = union {p in PROD} ORIG[p];       # All origins used by any product
set ALL_DEST = union {p in PROD} DEST[p];       # All destinations used by any product
set COMMON_ORIG = inter {p in PROD} ORIG[p];    # Origins common to ALL products (requires PROD non-empty)

#### *.dat Data [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/ch01_intro.ipynb)

In [ ]:
data;
set PROD = 'nut', 'popcorn', 'water';
set ORIG[nut]= 'US', 'Brazil';
set ORIG[popcorn] = 'US', 'Mexico';
set ORIG[water] = 'US', 'Canada', 'France';
set DEST [nut] = 'NY', 'LA';
set DEST [popcorn] = 'NY', 'Chicago';
set DEST [water] = 'NY', 'LA', 'Chicago'; 

display ALL_ORIG,   # US Brazil Mexico Canada France
ALL_DEST,           # NY LA Chicago
COMMON_ORIG;        # US

#### Python Data [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/ch01_intro.ipynb)

In [ ]:
PROD = {'nut', 'popcorn', 'water'}

ORIG = {
    'nut':     {'US', 'Brazil'},
    'popcorn': {'US', 'Mexico'},
    'water':   {'US', 'Canada', 'France'},}

DEST = {
    'nut':     {'NY', 'LA'},
    'popcorn': {'NY', 'Chicago'},
    'water':   {'NY', 'LA', 'Chicago'},}

ampl.set["PROD"] = PROD             # Assign entire set at once (OK)

for p in PROD:                      # Populate indexed sets per product (OK, but ensure ORIG/DEST are dicts)
    ampl.set["ORIG"][p] = ORIG[p]
    ampl.set["DEST"][p] = DEST[p]

# Fetch derived sets from AMPL and print in AMPL syntax
# Note: This assumes ALL_ORIG/ALL_DEST/COMMON_ORIG were already defined in the model
for name in ["ALL_ORIG", "ALL_DEST", "COMMON_ORIG"]:
    members = ampl.set[name].members()
    print(f"{name} := {' '.join(members)};")  # Avoid set() conversion (order loss)

# ALL_ORIG := US Brazil Mexico Canada France;
# ALL_DEST := NY LA Chicago;
# COMMON_ORIG := US;

#### Example 2 — All Four Algebraic Operators on Numeric Ranges [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/ch01_intro.ipynb)

This minimal example illustrates `union`, `inter`, `diff`, and `symdiff` using two
overlapping numeric ranges. The results can be verified by inspection.

In [ ]:
# Two overlapping sets of years (5-year intervals)
set Y1 = 2010 .. 2040 by 5;   # {2010, 2015, 2020, 2025, 2030, 2035, 2040}
set Y2 = 2020 .. 2045 by 5;   # {2020, 2025, 2030, 2035, 2040, 2045}

# Union: any year in Y1 or Y2
display Y1 union Y2;    # Expected: {2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045}

# Intersection: years in both Y1 and Y2
display Y1 inter Y2;    # Expected: {2020, 2025, 2030, 2035, 2040}

# Difference: years in Y1 but not Y2
display Y1 diff Y2;     # Expected: {2010, 2015}

# Symmetric difference: years in exactly one of Y1, Y2
display Y1 symdiff Y2;  # Expected: {2010, 2015, 2045}

#### Example 3 — Deriving an Index Set via `union` (Diet Model Pattern) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/ch01_intro.ipynb)

A standard AMPL idiom: derive a master index set as the union of two supplied sets, so
that the model never requires the user to maintain a redundant third set. This is the
canonical use case for `union` in optimization models (Fourer, Gay, Kernighan, Figure 5-1).

In [ ]:
# --- Model file ---
set MINREQ;                           # nutrients with a minimum daily requirement
set MAXREQ;                           # nutrients with a maximum daily requirement
set NUTR = MINREQ union MAXREQ;       # all nutrients; derived — not supplied in data
set FOOD;                             # foods

param cost  {FOOD}  > 0;              # cost per unit of each food
param f_min {FOOD}  >= 0;             # minimum servings of each food
param f_max {j in FOOD} >= f_min[j];  # maximum servings of each food
param n_min {MINREQ} >= 0;            # minimum nutrient amounts
param n_max {MAXREQ} >= 0;            # maximum nutrient amounts
param amt   {NUTR, FOOD} >= 0;        # nutrient amount per unit of food

var Buy {j in FOOD} >= f_min[j], <= f_max[j];

minimize TotalCost:
    sum {j in FOOD} cost[j] * Buy[j];

subject to DietMin {i in MINREQ}:
    sum {j in FOOD} amt[i,j] * Buy[j] >= n_min[i];

subject to DietMax {i in MAXREQ}:
    sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

In this model, only `MINREQ`, `MAXREQ`, and `FOOD` need to be specified in the data.
`NUTR` is derived automatically. If a nutrient appears in both `MINREQ` and `MAXREQ`,
it is included once in `NUTR` (set union discards duplicates), and both a minimum and
maximum constraint are generated for it.

#### Example 4 — Cartesian Product (`cross`) and Named Compound Sets [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/ch01_intro.ipynb)

The `cross` operator gives the set of all ordered pairs from two sets. In the
transportation model, naming this set `LINKS` clarifies intent and allows both
parameters and variables to be indexed over the same declared set.

In [ ]:
# Transportation model — naming the Cartesian product set
set ORIG;                             # origin cities
set DEST;                             # destination cities
set LINKS = ORIG cross DEST;          # all (origin, destination) pairs

param supply {ORIG} >= 0;             # supply available at each origin
param demand {DEST} >= 0;             # demand required at each destination
param cost   {LINKS} >= 0;           # cost per unit shipped on each link

check: sum {i in ORIG} supply[i] = sum {j in DEST} demand[j];

var Trans {LINKS} >= 0;              # units shipped on each link

minimize TotalCost:
    sum {(i,j) in LINKS} cost[i,j] * Trans[i,j];

subject to Supply {i in ORIG}:
    sum {j in DEST} Trans[i,j] = supply[i];

subject to Demand {j in DEST}:
    sum {i in ORIG} Trans[i,j] = demand[j];

**Equivalent indexing forms.** `ORIG cross DEST` and the comma-syntax indexing expression
`{ORIG, DEST}` denote the same mathematical object — the full Cartesian product.
They differ syntactically: `cross` is a binary operator producing a named set;
comma-syntax inside `{..., ...}` is an indexing expression used inline.

The declaration `set LINKS = ORIG cross DEST;` and the inline form
`sum {i in ORIG, j in DEST} ...` cover the same set of pairs. Use the named form when
multiple model components share the same index domain.

In [ ]:
# Three equivalent ways to express the same Cartesian product:

set ORIG;  
set DEST;
set LINKS = ORIG cross DEST;

param cost1 {LINKS} >= 0;               # Form 1: named set using cross operator
param cost2 {ORIG, DEST} >= 0;          # Form 2: anonymous comma-syntax in an indexing expression
param cost3 {i in ORIG, j in DEST} >= 0;# Form 3: named dummy indices in comma-syntax

# All three declare a parameter indexed over all (origin, destination) pairs.
# Forms 1 and 2 are preferred when the index set is referenced more than once.

#### Example 5 — Operator Precedence and Complex Set Expressions

Because `inter` binds more tightly than `union`, `diff`, and `symdiff`, compound
expressions can be written without parentheses — but parentheses are strongly recommended
for clarity. This example shows how precedence affects results.

In [ ]:
set A = {1, 2, 3, 4};
set B = {3, 4, 5, 6};
set C = {4, 5, 6, 7};
set D = {6, 7, 8};

display A union B inter C diff D;       # Precedence example: inter binds tighter than union and diff: { 1 2 3 4 5}
display (A union (B inter C)) diff D;   # Same expression with explicit parentheses for clarity: { 1 2 3 4 5}
display A symdiff (A symdiff B);        # symdiff identity: A symdiff (A symdiff B) = B

set Y1 = 2010 .. 2030 by 10;            # {2010, 2020, 2030}
display Y1 union {2035};                # Expected: {2010, 2020, 2030, 2035}


#### Example 6 — Subset (`within`) for Sparse Compound Sets

In multi-dimensional models, `within` on a declared set enforces that the data-supplied
pairs lie within the full Cartesian product. AMPL accepts **two equivalent syntactic forms**
for this constraint; both are common in AMPL literature.

In [ ]:
set ORIG;                                  # origins
set DEST;                                  # destinations

# Form 1: cross-operator form (explicit Cartesian product)
set LINKS within ORIG cross DEST;          # sparse subset of all (orig, dest) pairs

# Form 2: indexing-expression form (equivalent; used in Fourer et al. Fig 6-1)
set LINKS within {ORIG, DEST};           # same constraint, different syntax

param cost {LINKS} >= 0;                   # cost only for declared links
var Trans {LINKS} >= 0;                    # shipment only on declared links

<div class="alert alert-block alert-success"> <b>🛠 Best practice: </b><br>
Define derived index sets using <code>union</code>, <code>inter</code>, and similar operators rather than supplying them as separate data. This eliminates the risk of data inconsistency: if <code>NUTR</code> is defined as <code>MINREQ union MAXREQ</code>, it is always exactly the set of all nutrients — no manual synchronisation is required when <code>MINREQ</code> or <code>MAXREQ</code> change between instances.</div>

<div class="alert alert-block alert-success"> <b>🛠 Best practice: </b><br>
Declare sparse compound sets with <code>within ORIG cross DEST</code> (or the appropriate Cartesian product) rather than <code>dimen 2</code> alone. The <code>within</code> attribute causes AMPL to validate each data element against the component sets immediately at data-load time, producing an informative error rather than a silent wrong answer at solve time.</div>

<div class="alert alert-block alert-warning"> <b>⚠️ Pitfall: </b><br>
A scalar value is not a valid operand to a set operator — even when it is a member of the target set. Write <code>Y1 union {2045}</code>, not <code>Y1 union 2045</code>. The latter is a syntax error because <code>2045</code> is a number, not a set; it must be wrapped in braces to become the singleton set <code>{2045}</code>.</div>

<div class="alert alert-block alert-warning"> <b>⚠️ Pitfall: </b><br>
The operands of <code>union</code>, <code>inter</code>, <code>diff</code>, and <code>symdiff</code> must have the same dimension. Attempting to form the union of a 1-D set of strings with a 2-D set of pairs (e.g., <code>PROD union LINKS</code> where <code>LINKS</code> is a set of pairs) is a type error that AMPL reports at model-generation time. Use <code>cross</code> to build higher-dimensional sets from lower-dimensional ones.</div>

<div class="alert alert-block alert-warning"> <b>⚠️ Pitfall: </b><br>
Because <code>union</code>, <code>diff</code>, and <code>symdiff</code> share the same precedence and associate left-to-right, the expression <code>A diff B union C</code> means <code>(A diff B) union C</code>, which is typically not what is intended. Use explicit parentheses whenever mixing <code>union</code>, <code>diff</code>, and <code>symdiff</code> in the same expression.</div>

#### amplpy Equivalents

In amplpy, set operations are performed in AMPL (via `ampl.eval()`) or their results are
read back into Python. There are no direct amplpy methods corresponding to these operators `union`, `inter`, `diff`, `symdiff`, or `cross`; they are evaluated within AMPL.
Python's own set operations (`|`, `&`, `-`, `^`) on the result of `ampl.set[name].members()`
can replicate the logic, but the canonical approach is to define derived sets in the model
and retrieve them via amplpy.

| AMPL operation | amplpy equivalent |
|---|---|
| `set R = A union B;` then use `R` | Define in model via `ampl.eval(...)`, read back with `ampl.set['R'].members()` |
| `display A inter B;` | `ampl.eval("display A inter B;")` or `ampl.get_value('card(A inter B)')` |
| `A within B` (Boolean check) | `ampl.get_value('A within B')` returns 1 or 0 |
| `card(A union B)` | `ampl.get_value('card(A union B)')` |
| Assign members to `A` | `ampl.set['A'].set_values([...])` or `ampl.set_data(df, 'A')` |
| Read members of derived set `R` | `ampl.set['R'].members()` — returns a Python list |
| Read members as DataFrame | `ampl.set['R'].get_values().to_pandas()` |

**When to prefer Python set operations over AMPL set operators.**

If the set logic is needed only in Python (for example, to select which rows of a
DataFrame to send to AMPL), use Python's native `set` type:

```python
minreq = set(ampl.set['MINREQ'].members())
maxreq = set(ampl.set['MAXREQ'].members())
both = minreq & maxreq          # Python set intersection
```

If the result is needed **inside the AMPL model** — as an index set for a parameter,
variable, or constraint — define it in the model using AMPL operators so that AMPL
tracks all dependencies and updates the set automatically when data changes.

#### Edge Cases

**Edge case 1: Set operations on empty sets.**

All five operators are well-defined when one or both operands are empty. No error is raised.
The results follow directly from the mathematical definitions:

| Expression | Result when `B = {}` | Explanation |
|---|---|---|
| `A union {}` | `A` | Union with empty set is identity |
| `A inter {}` | `{}` | Intersection with empty set is empty |
| `A diff {}` | `A` | Removing nothing leaves `A` |
| `{} diff A` | `{}` | Nothing to remove from empty set |
| `A symdiff {}` | `A` | Symmetric diff with empty set is identity |
| `A cross {}` | `{}` | Product with empty set is empty |
| `{} within A` | `1` (true) | Empty set is a subset of every set |

The empty-set identity for `union` means that `set NUTR = MINREQ union MAXREQ;` is correct
even when `MAXREQ` is empty for a particular instance — `NUTR` will simply equal `MINREQ`.
No guard or special case is required.

In [ ]:
set MINREQ = {'A', 'B2', 'C', 'CAL'};
set MAXREQ = {};                        # empty for this instance
set NUTR   = MINREQ union MAXREQ;       # = MINREQ; no error
set BOTH   = MINREQ inter MAXREQ;       # = {}; no error

# A constraint indexed over BOTH generates zero instances -- not an error
param n_max {MINREQ} >= 0 default 0;
param n_min {MINREQ} >= 0 default 0;
# This subject to generates no constraints when BOTH is empty:
subject to BothBound {i in BOTH}:
    n_min[i] <= n_max[i];

display NUTR,   # {A, B2, C, CAL}
BOTH,           # {}
BothBound;      # ''

In [ ]:
set FOODS = {'bread', 'milk'};
set NUTRIENTS {FOODS};          # properly indexed set
set EMPTY_FOODS = {};           # truly empty
set BAD = inter {f in EMPTY_FOODS} NUTRIENTS[f];   # ❌ RUNTIME ERROR: empty iterator in iterated intersection

data;
set NUTRIENTS[bread] = 'A', 'B2';
set NUTRIENTS[milk]  = 'C', 'CAL';

display BAD;    # ❌ RUNTIME ERROR: empty iterator in iterated intersection

**Edge case 2: Operations on compound (multi-dimensional) sets.**

`union`, `inter`, `diff`, and `symdiff` apply to compound sets (sets of pairs, triples,
etc.) with the same semantics as for simple sets, provided both operands have the same
dimension. `cross` may be chained to build sets of arbitrary dimension.

A common use: take the union of two sparse arc sets that share the same component sets.

In [ ]:
set ORIG;  
set DEST;  
set PROD;

# Two sparse arc sets — both are subsets of ORIG cross DEST
set LINKS_A within ORIG cross DEST;
set LINKS_B within ORIG cross DEST;

set ALL_LINKS = LINKS_A union LINKS_B;  # Union of two 2-D sets yields a 2-D set
set SHARED   = LINKS_A inter LINKS_B;   # Intersection: arcs active in both networks

# Three-dimensional Cartesian product
set ROUTES = ORIG cross DEST cross PROD;# set of (orig, dest, prod) triples

param cost {ROUTES} >= 0;
var Ship  {ROUTES} >= 0;


minimize TotalCost:                     # Dummy indices in a sum over a 3-D set must be given as a triple
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Ship[i,j,p];

**Edge case 3: `cross` vs. comma-syntax — when they differ.**

`A cross B` and `{A, B}` produce the same set of pairs. However, only the comma-syntax
form allows the second set to **depend on the first dummy index**. This is the key
distinction for irregular (non-rectangular) index structures, such as node-specific arc
sets in a network.

In [ ]:
set NODES;                          # nodes in a graph
set ARCS {NODES};                   # ARCS[i] = set of nodes reachable from node i

# Correct: comma-syntax allows ARCS[i] to depend on the first dummy index i
param cap {i in NODES, j in ARCS[i]} >= 0;  # capacity of arc (i,j)
var Flow {i in NODES, j in ARCS[i]} >= 0, <= cap[i,j];

# 'cross' cannot express this: NODES cross ARCS is not valid because
# ARCS is an indexed collection of sets, not a single set.
# The following would be WRONG:
# set ALL_ARCS = NODES cross ARCS;   # ERROR: ARCS is not a set expression

# The correct named-set equivalent uses setof:
set ALL_ARCS = setof {i in NODES, j in ARCS[i]} (i,j);  # 2-D set of valid pairs

**Edge case 4: `within` on a symbolically defined set.**

When a set is defined with `=` in a model declaration, using `within` as a declaration
attribute on that set is redundant but not an error. Because the set is fully determined
by the expression, its membership is always consistent with any `within` constraint that
would be satisfied by the expression. However, if the expression produces elements outside
the `within` domain, AMPL will report an error at model-generation time.

In [ ]:
set NUTR;
set MINREQ within NUTR;            # attribute: MINREQ must be a subset of NUTR
set MAXREQ within NUTR;            # attribute: MAXREQ must be a subset of NUTR

# Derived set: the within attribute is enforced by construction
# (MINREQ union MAXREQ is always a subset of NUTR if MINREQ and MAXREQ are)
set NUTR2 = MINREQ union MAXREQ;   # no need for 'within NUTR' here

data;
set NUTR   = A B2 C CAL NA FIBER;
set MINREQ = A B2 C CAL;        # subset of NUTR
set MAXREQ = CAL NA FIBER;      # subset of NUTR

display NUTR2;          # A B2 C CAL NA FIBER